<a href="https://www.kaggle.com/code/nguyncanhda/fork-of-part-i-eda-lending-club-dataset-polars?scriptVersionId=308231733" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
"""
Lending Club – Accepted Loans | EDA Pipeline
Khóa luận Tốt nghiệp
─────────────────────────────────────────────
Bản sửa đổi: Thay robust_impute() bằng time_aware_impute() để tránh
Data Leakage theo thời gian (nhận xét của anh Nghĩa).
 
Ý tưởng cốt lõi của fix:
    • Sắp xếp DataFrame theo issue_d (tăng dần).
    • Với mỗi cột số: tính median theo từng PERIOD (tháng của issue_d).
    • Forward-fill median theo chiều thời gian → kỳ nào thiếu dữ liệu
      thì dùng median của KỲ TRƯỚC, KHÔNG dùng kỳ sau.
    • Backward-fill chỉ làm fallback cho các kỳ ĐẦU TIÊN chưa có
      dữ liệu nào (bất khả kháng; cần ghi chú rõ trong báo cáo).
    • 33 dòng có issue_d = null → fallback bằng global median (tính
      trước khi join để không bị ảnh hưởng bởi giá trị đã fill).
"""

'\nLending Club – Accepted Loans | EDA Pipeline\nKhóa luận Tốt nghiệp\n─────────────────────────────────────────────\nBản sửa đổi: Thay robust_impute() bằng time_aware_impute() để tránh\nData Leakage theo thời gian (nhận xét của anh Nghĩa).\n \nÝ tưởng cốt lõi của fix:\n    • Sắp xếp DataFrame theo issue_d (tăng dần).\n    • Với mỗi cột số: tính median theo từng PERIOD (tháng của issue_d).\n    • Forward-fill median theo chiều thời gian → kỳ nào thiếu dữ liệu\n      thì dùng median của KỲ TRƯỚC, KHÔNG dùng kỳ sau.\n    • Backward-fill chỉ làm fallback cho các kỳ ĐẦU TIÊN chưa có\n      dữ liệu nào (bất khả kháng; cần ghi chú rõ trong báo cáo).\n    • 33 dòng có issue_d = null → fallback bằng global median (tính\n      trước khi join để không bị ảnh hưởng bởi giá trị đã fill).\n'

Vấn đề cũ (robust_impute): df[col].median() tính trên toàn bộ 2.26 triệu dòng, khoản vay năm 2007 bị fill bằng median có chứa dữ liệu 2018 → data leakage.
Fix (time_aware_impute): Thay vì 1 median toàn cục, hàm mới tính median theo từng tháng (group_by("_period")), rồi forward-fill theo chiều thời gian. Khoản vay 3/2010 bị null sẽ chỉ được fill bằng median của ≤ 3/2010.

Hai fallback được ghi chú rõ trong code:
* **Backward-fill:** chỉ áp dụng cho các kỳ đầu tiên (2007 trở đi) chưa đủ dữ liệu để forward-fill — bất khả kháng, cần mention trong báo cáo.
* **Global median**: chỉ dùng cho đúng 33 dòng có issue_d = null, không gán được kỳ nào.

# I. Thư viện cần sử dụng

In [2]:
import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import re
import os
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import xgboost as xgb
import catboost as cb
import polars.selectors as cs
from datetime import datetime

In [3]:
#--- CẤU HÌNH HIỂN THỊ ---
pl.Config.set_tbl_rows(200)
pl.Config.set_tbl_cols(200)

polars.config.Config

# II. Xử lí Dữ liệu

## 2.1 Đọc dữ liệu

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 0. LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
ACCEPTED_PATH = "../input/lending-club/accepted_2007_to_2018Q4.csv.gz"
 
print("Đang đọc dữ liệu Accepted...")
acc_df = pl.read_csv(
    ACCEPTED_PATH,
    ignore_errors=True,
    try_parse_dates=True,
    infer_schema_length=10_000,
)

print(f"Accepted Shape: {acc_df.shape}")
print(acc_df.glimpse())

Đang đọc dữ liệu Accepted...
Accepted Shape: (2260701, 151)
Rows: 2260701
Columns: 151
$ id                                         <i64> 68407277, 68355089, 68341763, 66310712, 68476807, 68426831, 68476668, 67275481, 68466926, 68616873
$ member_id                                  <str> null, null, null, null, null, null, null, null, null, null
$ loan_amnt                                  <f64> 3600.0, 24700.0, 20000.0, 35000.0, 10400.0, 11950.0, 20000.0, 20000.0, 10000.0, 8000.0
$ funded_amnt                                <f64> 3600.0, 24700.0, 20000.0, 35000.0, 10400.0, 11950.0, 20000.0, 20000.0, 10000.0, 8000.0
$ funded_amnt_inv                            <f64> 3600.0, 24700.0, 20000.0, 35000.0, 10400.0, 11950.0, 20000.0, 20000.0, 10000.0, 8000.0
$ term                                       <str> ' 36 months', ' 36 months', ' 60 months', ' 60 months', ' 60 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months'
$ int_rate                                   <f64

## 2.2 Chuẩn bị tên cột để dễ xử lí

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. CHUẨN HÓA TÊN CỘT
# ─────────────────────────────────────────────────────────────────────────────
def normalize_cols(df: pl.DataFrame) -> pl.DataFrame:
    """Chuyển tên cột về chữ thường, thay khoảng trắng và dấu '-' bằng '_'."""
    new_cols = [
        c.lower().strip().replace(" ", "_").replace("-", "_")
        for c in df.columns
    ]
    return df.rename(dict(zip(df.columns, new_cols)))
 
 
acc_df = normalize_cols(acc_df)
print("Đã chuẩn hóa tên cột.")

Đã chuẩn hóa tên cột.


## 2.3 Xử lí đơn vị trong các cột

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. PHÁT HIỆN CỘT CHỨA ĐƠN VỊ ĐO
# ─────────────────────────────────────────────────────────────────────────────
def suggest_columns_with_units(df: pl.DataFrame, sample_n: int = 1_000) -> list[str]:
    """Tìm cột String có chuỗi dạng số + chữ (vd: '36 months', '10+ years')."""
    pattern = r"\d+\s*[a-zA-Z]+"
    return [
        col
        for col in df.select(cs.string()).columns
        if (
            sample := df.select(pl.col(col)).drop_nulls().head(sample_n)
        ).height > 0
        and sample.select(pl.col(col).str.contains(pattern).sum()).item() > 0
    ]
 
 
print("\nCác cột nghi ngờ có chứa đơn vị:")
print(suggest_columns_with_units(acc_df))


Các cột nghi ngờ có chứa đơn vị:
['term', 'emp_title', 'emp_length', 'loan_status', 'desc', 'zip_code', 'hardship_type', 'hardship_loan_status']


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. XỬ LÝ CỘT CHỨA ĐƠN VỊ ĐO & ĐỊNH DẠNG ĐẶC BIỆT
# ─────────────────────────────────────────────────────────────────────────────
def clean_units_and_format(df: pl.DataFrame) -> pl.DataFrame:
    """
    Tách đơn vị, chuẩn hoá kiểu dữ liệu cho các cột đặc thù:
        term         : ' 36 months' → 36  (Int32, tên mới: term_months)
        emp_length   : '10+ years'  → 10  (Int32, tên mới: emp_length_years)
        zip_code     : '123xx'      → '123' (String, tên mới: zip_code_prefix)
    """
    # term: ' 36 months' → 36
    if "term" in df.columns and df.schema["term"] == pl.String:
        df = df.with_columns(
            pl.col("term")
            .str.replace_all(r"[^\d]", "")
            .cast(pl.Int32, strict=False)
            .alias("term_months")
        ).drop("term")
 
    # emp_length: '< 1 year'→0, '10+ years'→10, 'n years'→n
    if "emp_length" in df.columns and df.schema["emp_length"] == pl.String:
        df = df.with_columns(
            pl.when(pl.col("emp_length").str.contains("< 1"))
            .then(pl.lit(0))
            .when(pl.col("emp_length").str.contains(r"10\+"))
            .then(pl.lit(10))
            .otherwise(pl.col("emp_length").str.extract(r"(\d+)", 1))
            .cast(pl.Int32, strict=False)
            .alias("emp_length_years")
        ).drop("emp_length")
 
    # zip_code: '123xx' → '123'
    if "zip_code" in df.columns and df.schema["zip_code"] == pl.String:
        df = df.with_columns(
            pl.col("zip_code")
            .str.replace_all("xx", "")
            .str.strip_chars()
            .alias("zip_code_prefix")
        ).drop("zip_code")
 
    return df

print("\nĐang xử lý đơn vị và định dạng...")
acc_df = clean_units_and_format(acc_df)
print("Hoàn thành.")


Đang xử lý đơn vị và định dạng...
Hoàn thành.


## 2.4 Xử lí Missing Values

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. KIỂM TRA & XÓA CỘT CÓ QUÁ NHIỀU GIÁ TRỊ THIẾU
# ─────────────────────────────────────────────────────────────────────────────
def check_missing(df: pl.DataFrame) -> pl.DataFrame:
    """Thống kê số lượng và tỉ lệ null của từng cột, sắp xếp giảm dần."""
    total_rows = df.height
    return (
        df.select([pl.col(c).null_count() for c in df.columns])
        .transpose(include_header=True, header_name="column_name")
        .rename({"column_0": "null_count"})
        .with_columns(
            (pl.col("null_count") / total_rows * 100).alias("null_percent")
        )
        .sort("null_percent", descending=True)
    )
 
 
print("\nBảng thống kê Missing Values:")
missing_info = check_missing(acc_df)
print(missing_info)
 
MISSING_THRESHOLD = 70.0
cols_to_drop = (
    missing_info
    .filter(pl.col("null_percent") > MISSING_THRESHOLD)
    .select("column_name")
    .to_series()
    .to_list()
)
print(f"\nĐang xóa {len(cols_to_drop)} cột có > {MISSING_THRESHOLD}% missing...")
acc_df = acc_df.drop(cols_to_drop)
print("Hoàn thành.")


Bảng thống kê Missing Values:
shape: (151, 3)
┌─────────────────────────────────┬────────────┬──────────────┐
│ column_name                     ┆ null_count ┆ null_percent │
│ ---                             ┆ ---        ┆ ---          │
│ str                             ┆ u32        ┆ f64          │
╞═════════════════════════════════╪════════════╪══════════════╡
│ member_id                       ┆ 2260701    ┆ 100.0        │
│ orig_projected_additional_accr… ┆ 2252050    ┆ 99.617331    │
│ hardship_type                   ┆ 2249784    ┆ 99.517097    │
│ hardship_reason                 ┆ 2249784    ┆ 99.517097    │
│ hardship_status                 ┆ 2249784    ┆ 99.517097    │
│ deferral_term                   ┆ 2249784    ┆ 99.517097    │
│ hardship_amount                 ┆ 2249784    ┆ 99.517097    │
│ hardship_start_date             ┆ 2249784    ┆ 99.517097    │
│ hardship_end_date               ┆ 2249784    ┆ 99.517097    │
│ payment_plan_start_date         ┆ 2249784    ┆ 99.51709

**Note**: 70% là mốc tự chọn

## **2.5 Xử lí các cột về Thời gian**

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. XỬ LÝ NGÀY THÁNG & FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
DATE_COLS = ["earliest_cr_line", "last_pymnt_d", "last_credit_pull_d", "issue_d"]
 
 
def process_date_columns(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:
    """Chuyển cột String dạng 'Mon-YYYY' sang kiểu pl.Date."""
    for col in cols:
        if col not in df.columns:
            continue
        dtype = df.schema[col]
        if dtype == pl.String:
            print(f"  Đang chuyển: {col}")
            df = df.with_columns(
                pl.col(col).str.to_date(format="%b-%Y", strict=False).alias(col)
            )
        else:
            print(f"  Bỏ qua: {col} (đã là {dtype})")
    return df
 
 
print("\nĐang chuyển đổi định dạng ngày tháng...")
acc_df = process_date_columns(acc_df, DATE_COLS)
 
# Feature: credit_history_months = khoảng cách (tháng) giữa issue_d và earliest_cr_line
if "earliest_cr_line" in acc_df.columns and "issue_d" in acc_df.columns:
    acc_df = acc_df.with_columns(
        (
            (pl.col("issue_d").dt.year() - pl.col("earliest_cr_line").dt.year()) * 12
            + (pl.col("issue_d").dt.month() - pl.col("earliest_cr_line").dt.month())
        ).alias("credit_history_months")
    )
 
# Feature: days_since_<col> = số ngày từ ref_date đến thời điểm cuối cùng
REF_DATE = datetime(2019, 1, 1)
for col in ["last_pymnt_d", "last_credit_pull_d"]:
    if col in acc_df.columns:
        acc_df = acc_df.with_columns(
            (pl.lit(REF_DATE).cast(pl.Date) - pl.col(col))
            .dt.total_days()
            .alias(f"days_since_{col}")
        )
 
print(
    acc_df.select(
        ["issue_d", "credit_history_months", "days_since_last_pymnt_d"]
    ).head()
)


Đang chuyển đổi định dạng ngày tháng...
  Đang chuyển: earliest_cr_line
  Đang chuyển: last_pymnt_d
  Đang chuyển: last_credit_pull_d
  Đang chuyển: issue_d
shape: (5, 3)
┌────────────┬───────────────────────┬─────────────────────────┐
│ issue_d    ┆ credit_history_months ┆ days_since_last_pymnt_d │
│ ---        ┆ ---                   ┆ ---                     │
│ date       ┆ i32                   ┆ i64                     │
╞════════════╪═══════════════════════╪═════════════════════════╡
│ 2015-12-01 ┆ 148                   ┆ 0                       │
│ 2015-12-01 ┆ 192                   ┆ 944                     │
│ 2015-12-01 ┆ 184                   ┆ 579                     │
│ 2015-12-01 ┆ 87                    ┆ -31                     │
│ 2015-12-01 ┆ 210                   ┆ 914                     │
└────────────┴───────────────────────┴─────────────────────────┘


## **2.6 Xử lí Data Leakage về Thời gian**

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. IMPUTATION KHÔNG GÂY DATA LEAKAGE (TIME-AWARE)
# ─────────────────────────────────────────────────────────────────────────────
def time_aware_impute(df: pl.DataFrame, time_col: str = "issue_d") -> pl.DataFrame:
    """
    Impute giá trị null theo nhóm thời gian để tránh Data Leakage.
 
    Chiến lược cho cột SỐ:
    ─────────────────────────────────────────────────────────────────────────
    Vấn đề của median toàn cột (cách cũ):
        Khoản vay ngày 1/1/2007 bị null → fill bằng median tính cả dữ liệu
        2018 → dữ liệu tương lai "rò rỉ" vào quá khứ = DATA LEAKAGE.
 
    Cách sửa (time-aware):
        1. Sắp xếp theo time_col (tăng dần).
        2. Tính median PER PERIOD (theo tháng của issue_d).
        3. Forward-fill median: kỳ thiếu → dùng median kỳ TRƯỚC.
           → Không bao giờ dùng thông tin từ tương lai.
        4. Backward-fill (fallback): chỉ dùng cho các kỳ ĐẦU TIÊN
           chưa có dữ liệu để forward-fill (cần ghi chú trong báo cáo).
        5. Fallback cho 33 dòng null issue_d: global median tính
           TRƯỚC khi join (chỉ trên dữ liệu gốc, không leakage thêm).
 
    Chiến lược cho cột CHỮ / BOOLEAN:
        → Điền "Unknown" / False (không phụ thuộc thời gian, không leakage).
    ─────────────────────────────────────────────────────────────────────────
    """
    total_start = df.select(pl.all().null_count()).sum_horizontal().item()
    print(f"\nTổng missing value ban đầu: {total_start:,}")
 
    # ── Xác định cột cần impute ──────────────────────────────────────────────
    numeric_with_null = [
        c for c in df.select(cs.numeric()).columns
        if c != time_col and df[c].null_count() > 0
    ]
    cat_with_null = [
        c for c in df.select(cs.string() | cs.categorical() | cs.boolean()).columns
        if df[c].null_count() > 0 and c != time_col
    ]
    print(
        f"  Cột số cần impute: {len(numeric_with_null)} | "
        f"Cột chữ/boolean: {len(cat_with_null)}"
    )
 
    # ── Tính global median TRƯỚC KHI join (fallback cho null time_col) ──────
    # Vẫn là median toàn cột, nhưng chỉ dùng cho ~33 dòng không có issue_d.
    # Ghi chú: đây là trường hợp bất khả kháng, sẽ ghi rõ trong báo cáo.
    print("  Tính global median (dùng làm fallback cuối cùng)...")
    global_medians: dict[str, float] = {
        c: (df[c].median() or 0.0) for c in numeric_with_null
    }
 
    # ── Sắp xếp theo thời gian ───────────────────────────────────────────────
    df = df.sort(time_col, nulls_last=True)
 
    # ── Tạo cột period (năm-tháng, bỏ qua null) ─────────────────────────────
    df = df.with_columns(
        pl.col(time_col).dt.truncate("1mo").alias("_period")
    )
 
    # ── Tính median theo từng period ─────────────────────────────────────────
    print("  Tính period median...")
    period_medians = (
        df
        .group_by("_period")
        .agg([pl.col(c).median().alias(f"_med_{c}") for c in numeric_with_null])
        .sort("_period")
    )
 
    # ── Forward-fill rồi backward-fill ───────────────────────────────────────
    # Forward-fill: kỳ T thiếu → dùng median kỳ T-1 (không dùng tương lai)
    # Backward-fill: chỉ xử lý kỳ đầu tiên chưa có dữ liệu (ghi chú báo cáo)
    med_cols = [c for c in period_medians.columns if c.startswith("_med_")]
    period_medians = (
        period_medians
        .with_columns([pl.col(c).forward_fill() for c in med_cols])
        .with_columns([pl.col(c).backward_fill() for c in med_cols])
    )
 
    # ── Join median về DataFrame chính ───────────────────────────────────────
    df = df.join(period_medians, on="_period", how="left")
 
    # ── Điền null bằng period median, rồi global median làm fallback ─────────
    fill_num_exprs = [
        pl.col(c)
        .fill_null(pl.col(f"_med_{c}"))   # Ưu tiên: period median (time-aware)
        .fill_null(global_medians[c])       # Fallback: global median (chỉ ~33 dòng)
        .alias(c)
        for c in numeric_with_null
        if f"_med_{c}" in df.columns
    ]
    if fill_num_exprs:
        df = df.with_columns(fill_num_exprs)
 
    # ── Xóa cột helper ───────────────────────────────────────────────────────
    drop_helper = ["_period"] + [c for c in df.columns if c.startswith("_med_")]
    df = df.drop(drop_helper)
 
    # ── Điền null cột chữ / boolean ──────────────────────────────────────────
    fill_cat_exprs = [
        pl.col(c).fill_null(False if df.schema[c] == pl.Boolean else "Unknown")
        for c in cat_with_null
        if c in df.columns
    ]
    if fill_cat_exprs:
        df = df.with_columns(fill_cat_exprs)
 
    total_end = df.select(pl.all().null_count()).sum_horizontal().item()
    print(f"Tổng missing value sau imputation: {total_end:,}")
    print("Time-aware imputation hoàn thành.")
    return df
 
 
print("\nĐang thực hiện time-aware imputation...")
acc_df = time_aware_impute(acc_df, time_col="issue_d")


Đang thực hiện time-aware imputation...

Tổng missing value ban đầu: 19,429,417
  Cột số cần impute: 91 | Cột chữ/boolean: 18
  Tính global median (dùng làm fallback cuối cùng)...
  Tính period median...
Tổng missing value sau imputation: 2,660
Time-aware imputation hoàn thành.


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. KIỂM TRA GIÁ TRỊ UNIQUE
# ─────────────────────────────────────────────────────────────────────────────
def check_unique(df: pl.DataFrame, top_n: int = 5) -> pl.DataFrame:
    """Thống kê số lượng giá trị unique và mẫu đại diện của từng cột."""
    return pl.DataFrame(
        [
            (col, df[col].n_unique(), str(df[col].unique().to_list()[:top_n]))
            for col in df.columns
        ],
        schema=["Column", "N_Unique", f"Sample_Values_{top_n}"],
    )
 
 
print("\nKiểm tra giá trị Unique:")
print(check_unique(acc_df))


Kiểm tra giá trị Unique:
shape: (113, 3)
┌────────────────────────────────┬──────────┬─────────────────────────────────┐
│ Column                         ┆ N_Unique ┆ Sample_Values_5                 │
│ ---                            ┆ ---      ┆ ---                             │
│ str                            ┆ i64      ┆ str                             │
╞════════════════════════════════╪══════════╪═════════════════════════════════╡
│ id                             ┆ 2260669  ┆ [54734.0, 55521.0, 55716.0, 55… │
│ loan_amnt                      ┆ 1572     ┆ [500.0, 550.0, 600.0, 700.0, 7… │
│ funded_amnt                    ┆ 1572     ┆ [500.0, 550.0, 600.0, 700.0, 7… │
│ funded_amnt_inv                ┆ 10057    ┆ [0.0, 0.00012109810799999999, … │
│ int_rate                       ┆ 673      ┆ [5.31, 5.32, 5.42, 5.79, 5.93]  │
│ installment                    ┆ 93301    ┆ [4.93, 7.61, 14.01, 14.77, 15.… │
│ grade                          ┆ 8        ┆ ['E', 'C', 'Unknown', 'D', 'F'… 

/tmp/ipykernel_17/3713130940.py:6: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  return pl.DataFrame(


In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# [3] THAM CHIẾU TÀI LIỆU & GIẢI THÍCH KỸ THUẬT
# ═══════════════════════════════════════════════════════════════════════════════
DOCUMENTATION = """
╔══════════════════════════════════════════════════════════════════════════════╗
║  GIẢI THÍCH KỸ THUẬT: TIME-AWARE IMPUTATION                               ║
║  Kèm tài liệu tham khảo để trình bày trong Khóa luận Tốt nghiệp          ║
╚══════════════════════════════════════════════════════════════════════════════╝
 
1. VẤN ĐỀ: DATA LEAKAGE QUA IMPUTATION
────────────────────────────────────────────────────────────────────────────────
Khi một biến X tại thời điểm t bị missing và ta fill bằng median của toàn cột
(tính gộp từ t=2007 đến t=2018), thông tin từ TƯƠNG LAI (t > t_now) đã "rò rỉ"
vào điểm dữ liệu tại t. Điều này vi phạm nguyên tắc causal ordering trong
time-series machine learning.
 
Kaufman et al. (2012) định nghĩa đây là dạng "leakage through preprocessing":
  "Leakage introduces information about the response that would not be
   legitimately available to a model at prediction time."
  — Kaufman, S., Rosset, S., Perlich, C., & Stitelman, O. (2012).
    Leakage in Data Mining: Formulation, Detection, and Avoidance.
    ACM Transactions on Knowledge Discovery from Data, 6(4), 1–21.
    https://doi.org/10.1145/2382577.2382579
 
2. GIẢI PHÁP: EXPANDING WINDOW MEDIAN (Time-aware Imputation)
────────────────────────────────────────────────────────────────────────────────
Phương pháp triển khai trong code trên thuộc dạng "rolling/expanding statistics
for imputation". Ý tưởng xuất phát từ tài liệu time-series cross-validation:
 
  Nguyên tắc: Tại thời điểm t, chỉ được dùng thông tin từ [0, t] để impute.
 
  Cụ thể:
    (a) Sort dữ liệu theo issue_d (tăng dần): đảm bảo thứ tự thời gian.
    (b) Group by period (tháng): tính median cho từng tháng.
    (c) Forward-fill: kỳ T không có median → dùng median kỳ T-1.
        Không bao giờ dùng kỳ T+1, T+2, ... → không leakage.
    (d) Backward-fill (fallback duy nhất): chỉ cho kỳ ĐẦU TIÊN.
        Đây là bất khả kháng và phải ghi chú trong báo cáo.
 
Cách tiếp cận này tương đương với "expanding window imputation" được đề cập
trong các tài liệu về time-series forecasting:
 
  Bergmeir, C., Hyndman, R. J., & Koo, B. (2018).
  A note on the validity of cross-validation for evaluating autoregressive
  time series prediction. Computational Statistics & Data Analysis, 120, 70–83.
  https://doi.org/10.1016/j.csda.2017.11.003
  → Chứng minh rằng mọi bước preprocessing phải nằm trong fold tương ứng.
 
  Cerqueira, V., Torgo, L., & Mozetič, I. (2020).
  Evaluating time series forecasting models: An empirical study on performance
  estimation methods. Machine Learning, 109, 1997–2028.
  https://doi.org/10.1007/s10994-020-05910-7
  → Mục 3.2 thảo luận về contamination qua global preprocessing.
 
3. TẠI SAO KHÔNG DÙNG GROUP-BY TOÀN CỘT?
────────────────────────────────────────────────────────────────────────────────
Giả sử ta group_by(grade).agg(median) rồi fill:
  → Khoản vay Grade C tháng 1/2007 được fill bằng median Grade C
    của toàn bộ 2007–2018.
  → Vẫn bị leakage! Chỉ ít hơn về magnitude nhưng vẫn vi phạm về mặt lý thuyết.
 
Phương pháp đúng phải đảm bảo: "the imputed value for observation i
uses only data from observations j where t_j ≤ t_i."
  — Paparrizos, J., Liu, B., Elmore, A. J., & Franklin, M. J. (2022).
    IMDPUTE: Effective Imputation for Time Series.
    VLDB Endowment, 15(8).
 
4. THỰC TIỄN TRONG CÁC NGHIÊN CỨU LENDING CLUB
────────────────────────────────────────────────────────────────────────────────
Một số nghiên cứu đáng chú ý về Lending Club đã xử lý vấn đề này:
 
  • Byanjankar, A., Heikkilä, M., & Mezei, J. (2015).
    Predicting Credit Risk in Peer-to-Peer Lending: A Neural Network Approach.
    IEEE SSCI 2015.
    → Loại bỏ các biến post-origination trước khi training.
 
  • Serrano-Cinca, C., Gutiérrez-Nieto, B., & López-Palacios, L. (2015).
    Determinants of Default in P2P Lending.
    PLOS ONE, 10(10), e0139427.
    https://doi.org/10.1371/journal.pone.0139427
    → Chỉ dùng thông tin tại thời điểm ký hợp đồng (origination features).
 
  • Emekter, R., Tu, Y., Jirasakuldech, B., & Lu, M. (2015).
    Evaluating credit risk and loan performance in online Peer-to-Peer (P2P)
    lending. Applied Economics, 47(1), 54–70.
    → Thảo luận rõ về việc tách train/test theo thời gian để tránh leakage.
 
5. BEST PRACTICES CHO KHÓA LUẬN
────────────────────────────────────────────────────────────────────────────────
Trong phần Methodology của Khóa luận, nên trình bày:
 
  (1) Xác định "prediction horizon": Mô hình được train để dự đoán tại thời
      điểm nào? (Thường là tại issue_d — thời điểm phê duyệt khoản vay).
 
  (2) Feature eligibility rule: "Chỉ sử dụng thông tin có thể biết tại hoặc
      trước thời điểm issue_d."
 
  (3) Imputation policy: "Impute missing values sử dụng thống kê từ dữ liệu
      cùng kỳ hoặc trước đó (time-aware median imputation) để tránh leakage."
 
  (4) Ghi chú backward-fill: "33 dòng thiếu issue_d và các kỳ đầu tiên
      được xử lý bằng backward-fill hoặc global median — phương án bất khả
      kháng, ảnh hưởng < 0.01% tổng dữ liệu."
"""
 
print("\n" + "=" * 60)
print("[3] TÀI LIỆU THAM KHẢO & GIẢI THÍCH")
print("=" * 60)
print(DOCUMENTATION)


[3] TÀI LIỆU THAM KHẢO & GIẢI THÍCH

╔══════════════════════════════════════════════════════════════════════════════╗
║  GIẢI THÍCH KỸ THUẬT: TIME-AWARE IMPUTATION                               ║
║  Kèm tài liệu tham khảo để trình bày trong Khóa luận Tốt nghiệp          ║
╚══════════════════════════════════════════════════════════════════════════════╝
 
1. VẤN ĐỀ: DATA LEAKAGE QUA IMPUTATION
────────────────────────────────────────────────────────────────────────────────
Khi một biến X tại thời điểm t bị missing và ta fill bằng median của toàn cột
(tính gộp từ t=2007 đến t=2018), thông tin từ TƯƠNG LAI (t > t_now) đã "rò rỉ"
vào điểm dữ liệu tại t. Điều này vi phạm nguyên tắc causal ordering trong
time-series machine learning.
 
Kaufman et al. (2012) định nghĩa đây là dạng "leakage through preprocessing":
  "Leakage introduces information about the response that would not be
   legitimately available to a model at prediction time."
  — Kaufman, S., Rosset, S., Perlich, C., & Stitelman

In [13]:
#Xóa trùng lặp hoàn toàn
print(f"Số dòng trước khi drop duplicates: {acc_df.height}")
acc_df = acc_df.unique()
print(f"Số dòng sau khi drop duplicates: {acc_df.height}")

Số dòng trước khi drop duplicates: 2260701
Số dòng sau khi drop duplicates: 2260669


In [14]:
 
# ─────────────────────────────────────────────────────────────────────────────
# 8. PHÁT HIỆN OUTLIER (IQR)
# ─────────────────────────────────────────────────────────────────────────────
def scan_all_outliers(df: pl.DataFrame, min_unique: int = 10) -> pl.DataFrame:
    """
    Phát hiện outlier bằng phương pháp IQR cho các cột số.
    Chỉ kiểm tra các cột có trên min_unique giá trị khác nhau.
    """
    numeric_cols = [
        c for c in df.select(cs.numeric()).columns
        if df[c].n_unique() > min_unique
    ]
    print(f"Đang quét outlier cho {len(numeric_cols)} cột số...")
 
    stats_list = []
    for col in numeric_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        if q1 is None or q3 is None:
            continue
        iqr = q3 - q1
        if iqr == 0:
            continue
        lb = q1 - 1.5 * iqr
        ub = q3 + 1.5 * iqr
        n_out = df.filter((pl.col(col) < lb) | (pl.col(col) > ub)).height
        if n_out > 0:
            stats_list.append(
                {
                    "Column": col,
                    "Outlier_Count": n_out,
                    "Percent (%)": round(n_out / df.height * 100, 2),
                    "Lower_Bound": round(lb, 2),
                    "Upper_Bound": round(ub, 2),
                    "Min_Val": df[col].min(),
                    "Max_Val": df[col].max(),
                }
            )
 
    return pl.DataFrame(stats_list).sort("Percent (%)", descending=True)

In [15]:
# Chạy lại
outlier_report = scan_all_outliers(acc_df)
print(outlier_report.head(15))

Đang quét outlier cho 86 cột số...
shape: (15, 7)
┌─────────────────┬───────────────┬─────────────┬─────────────┬─────────────┬─────────┬────────────┐
│ Column          ┆ Outlier_Count ┆ Percent (%) ┆ Lower_Bound ┆ Upper_Bound ┆ Min_Val ┆ Max_Val    │
│ ---             ┆ ---           ┆ ---         ┆ ---         ┆ ---         ┆ ---     ┆ ---        │
│ str             ┆ i64           ┆ f64         ┆ f64         ┆ f64         ┆ f64     ┆ f64        │
╞═════════════════╪═══════════════╪═════════════╪═════════════╪═════════════╪═════════╪════════════╡
│ il_util         ┆ 777565        ┆ 34.4        ┆ 62.5        ┆ 82.5        ┆ 0.0     ┆ 1000.0     │
│ mths_since_last ┆ 690380        ┆ 30.54       ┆ 20.0        ┆ 44.0        ┆ 0.0     ┆ 226.0      │
│ _delinq         ┆               ┆             ┆             ┆             ┆         ┆            │
│ mths_since_rece ┆ 601481        ┆ 26.61       ┆ 25.0        ┆ 41.0        ┆ 0.0     ┆ 202.0      │
│ nt_revol_delinq ┆               ┆      

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. XỬ LÝ OUTLIER (WINSORIZATION / CAPPING)
# ─────────────────────────────────────────────────────────────────────────────
def cap_outliers(
    df: pl.DataFrame, report: pl.DataFrame, threshold_pct: float = 5.0
) -> pl.DataFrame:
    """
    Capping outlier (Winsorization) cho các cột có tỷ lệ outlier < threshold_pct%.
    Các cột có outlier nhiều hơn ngưỡng sẽ được xem xét riêng (giữ nguyên hoặc
    transform log nếu cần thiết cho mô hình).
    """
    cols_to_cap = report.filter(pl.col("Percent (%)") < threshold_pct)
    exprs = [
        pl.col(row["Column"])
        .clip(row["Lower_Bound"], row["Upper_Bound"])
        .alias(row["Column"])
        for row in cols_to_cap.iter_rows(named=True)
    ]
    print(f"Đang capping {len(exprs)} cột (outlier < {threshold_pct}%)...")
    if exprs:
        df = df.with_columns(exprs)
    return df
 
 
print("\nĐang quét outlier...")
outlier_report = scan_all_outliers(acc_df)
print(outlier_report)
 
print("\nĐang capping outlier...")
acc_df = cap_outliers(acc_df, outlier_report, threshold_pct=5.0)
print("Hoàn thành xử lý outlier.")


Đang quét outlier...
Đang quét outlier cho 86 cột số...
shape: (68, 7)
┌───────────────┬───────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────────┐
│ Column        ┆ Outlier_Count ┆ Percent (%) ┆ Lower_Bound ┆ Upper_Bound ┆ Min_Val ┆ Max_Val      │
│ ---           ┆ ---           ┆ ---         ┆ ---         ┆ ---         ┆ ---     ┆ ---          │
│ str           ┆ i64           ┆ f64         ┆ f64         ┆ f64         ┆ f64     ┆ f64          │
╞═══════════════╪═══════════════╪═════════════╪═════════════╪═════════════╪═════════╪══════════════╡
│ il_util       ┆ 777565        ┆ 34.4        ┆ 62.5        ┆ 82.5        ┆ 0.0     ┆ 1000.0       │
│ mths_since_la ┆ 690380        ┆ 30.54       ┆ 20.0        ┆ 44.0        ┆ 0.0     ┆ 226.0        │
│ st_delinq     ┆               ┆             ┆             ┆             ┆         ┆              │
│ mths_since_re ┆ 601481        ┆ 26.61       ┆ 25.0        ┆ 41.0        ┆ 0.0     ┆ 202.0        │
│ cent_revol_de ┆  

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# KIỂM TRA CUỐI
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Shape cuối cùng: {acc_df.shape}")
remaining_null = acc_df.select(pl.all().null_count()).sum_horizontal().item()
print(f"Tổng null còn lại: {remaining_null:,}")
print(f"{'='*60}")
print(acc_df.head(3))


Shape cuối cùng: (2260669, 113)
Tổng null còn lại: 2,532
shape: (3, 113)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ id  ┆ loa ┆ fun ┆ fun ┆ int ┆ ins ┆ gra ┆ sub ┆ emp ┆ hom ┆ ann ┆ ver ┆ iss ┆ loa ┆ pym ┆ url ┆ pur ┆ tit ┆ add ┆ dti ┆ del ┆ ear ┆ fic ┆ fic ┆ inq ┆ mth ┆ ope ┆ pub ┆ rev ┆ rev ┆ tot ┆ ini ┆ out ┆ out ┆ tot ┆ tot ┆ tot ┆ tot ┆ tot ┆ rec ┆ col 

# Xử lí Null và Leakage ở các cột còn lại

In [18]:
"""
Lending Club – EDA Pipeline (Phần bổ sung)
═══════════════════════════════════════════════════════════════════════════════
Xử lý 3 vấn đề còn lại sau pipeline chính:
 
  [1] Xử lý 2,532 null còn lại
  [2] Phát hiện & loại bỏ các cột gây Data Leakage (target / temporal)
  [3] Giải thích chi tiết kỹ thuật Time-aware Imputation kèm tài liệu tham khảo
═══════════════════════════════════════════════════════════════════════════════
"""

'\nLending Club – EDA Pipeline (Phần bổ sung)\n═══════════════════════════════════════════════════════════════════════════════\nXử lý 3 vấn đề còn lại sau pipeline chính:\n \n  [1] Xử lý 2,532 null còn lại\n  [2] Phát hiện & loại bỏ các cột gây Data Leakage (target / temporal)\n  [3] Giải thích chi tiết kỹ thuật Time-aware Imputation kèm tài liệu tham khảo\n═══════════════════════════════════════════════════════════════════════════════\n'

In [19]:
# ═══════════════════════════════════════════════════════════════════════════════
# [1] CHẨN ĐOÁN VÀ XỬ LÝ 2,532 NULL CÒN LẠI
# ═══════════════════════════════════════════════════════════════════════════════
"""
Nguyên nhân:
──────────────────────────────────────────────────────────────────────────────
Sau time_aware_impute(), các cột kiểu DATE (pl.Date) KHÔNG được xử lý vì
cs.numeric() không chọn chúng. Các cột phái sinh từ date cũng bị null theo:
 
  • earliest_cr_line  → còn null
  • last_pymnt_d      → còn null  
  • last_credit_pull_d → còn null
  • credit_history_months   ← phái sinh, null khi earliest_cr_line null
  • days_since_last_pymnt_d ← phái sinh, null khi last_pymnt_d null
  • days_since_last_credit_pull_d ← tương tự
 
Chiến lược xử lý:
  • Cột DATE gốc (earliest_cr_line, last_pymnt_d, last_credit_pull_d):
    KHÔNG fill trực tiếp vì chúng đã được dùng để tạo feature phái sinh.
    Nếu fill bằng giá trị giả → feature phái sinh sẽ sai hơn.
    → Để nguyên NULL ở cột date gốc, chỉ fill ở feature phái sinh.
 
  • Cột NUMERIC phái sinh từ date:
    → Fill bằng median của cột đó (theo issue_d period, time-aware).
    → Đây là phương pháp phù hợp vì chúng là số nguyên/thực.
"""
 
 
def diagnose_remaining_nulls(df: pl.DataFrame) -> pl.DataFrame:
    """Chẩn đoán chi tiết các cột còn null sau imputation chính."""
    total_rows = df.height
    null_cols = [
        c for c in df.columns if df[c].null_count() > 0
    ]
    if not null_cols:
        print("Không còn null nào.")
        return pl.DataFrame()
 
    report = pl.DataFrame(
        [
            {
                "column": c,
                "dtype": str(df.schema[c]),
                "null_count": df[c].null_count(),
                "null_percent": round(df[c].null_count() / total_rows * 100, 4),
            }
            for c in null_cols
        ]
    ).sort("null_count", descending=True)
 
    print(f"Còn {len(null_cols)} cột có null ({df.select(pl.all().null_count()).sum_horizontal().item():,} giá trị):")
    print(report)
    return report

In [20]:
print("=" * 60)
print("[1] CHẨN ĐOÁN NULL CÒN LẠI")
print("=" * 60)
null_report = diagnose_remaining_nulls(acc_df)

[1] CHẨN ĐOÁN NULL CÒN LẠI
Còn 4 cột có null (2,532 giá trị):
shape: (4, 4)
┌────────────────────┬───────┬────────────┬──────────────┐
│ column             ┆ dtype ┆ null_count ┆ null_percent │
│ ---                ┆ ---   ┆ ---        ┆ ---          │
│ str                ┆ str   ┆ i64        ┆ f64          │
╞════════════════════╪═══════╪════════════╪══════════════╡
│ last_pymnt_d       ┆ Date  ┆ 2428       ┆ 0.1074       │
│ last_credit_pull_d ┆ Date  ┆ 73         ┆ 0.0032       │
│ earliest_cr_line   ┆ Date  ┆ 30         ┆ 0.0013       │
│ issue_d            ┆ Date  ┆ 1          ┆ 0.0          │
└────────────────────┴───────┴────────────┴──────────────┘


In [21]:
def fix_remaining_nulls(df: pl.DataFrame) -> pl.DataFrame:
    """
    Xử lý dứt điểm các null còn lại sau time_aware_impute().
 
    Phân loại và chiến lược:
    ─────────────────────────────────────────────────────────────────────────
    Cột DATE gốc (earliest_cr_line, last_pymnt_d, last_credit_pull_d):
        → KHÔNG fill. Giữ nguyên null để tránh tạo ra ngày giả.
          Các mô hình sẽ nhận giá trị null từ feature phái sinh (đã fill ở dưới).
 
    Cột NUMERIC phái sinh từ date:
        → Fill bằng median của cột đó (không có leakage vì đây là
          post-feature-engineering, không còn phụ thuộc thời gian trực tiếp).
          Lý do median: phân phối lệch (skewed), median robust hơn mean.
 
    Cột STRING/CATEGORICAL còn null:
        → Fill "Unknown" (trường hợp bỏ sót từ bước trước).
 
    Cột NUMERIC còn null (bỏ sót từ bước trước, edge case):
        → Fill bằng median toàn cột (lượng nhỏ, ảnh hưởng không đáng kể).
    ─────────────────────────────────────────────────────────────────────────
    """
    DATE_COLS_RAW = ["earliest_cr_line", "last_pymnt_d", "last_credit_pull_d", "issue_d"]
 
    # ── Cột DATE GỐC: Không fill, ghi nhận lại ───────────────────────────────
    date_nulls = [c for c in DATE_COLS_RAW if c in df.columns and df[c].null_count() > 0]
    if date_nulls:
        print(f"\n  Giữ nguyên null ở {len(date_nulls)} cột date gốc: {date_nulls}")
        print("  (Null này vô hại vì các feature phái sinh từ chúng sẽ được fill riêng)")
 
    # ── Cột NUMERIC phái sinh từ date: Fill bằng median ──────────────────────
    derived_date_cols = [
        c for c in df.columns
        if c.startswith("days_since_") or c == "credit_history_months"
    ]
    exprs_derived = []
    for c in derived_date_cols:
        if c in df.columns and df[c].null_count() > 0:
            med = df[c].median()
            if med is None:
                med = 0.0
            exprs_derived.append(pl.col(c).fill_null(med).alias(c))
            print(f"  Fill '{c}': {df[c].null_count():,} null → median = {med:.1f}")
 
    if exprs_derived:
        df = df.with_columns(exprs_derived)
 
    # ── Cột NUMERIC còn null khác (edge case) ─────────────────────────────────
    other_numeric_nulls = [
        c for c in df.select(cs.numeric()).columns
        if df[c].null_count() > 0
        and c not in DATE_COLS_RAW
        and not c.startswith("days_since_")
        and c != "credit_history_months"
    ]
    if other_numeric_nulls:
        print(f"\n  Các cột số còn null khác: {other_numeric_nulls}")
        exprs_num = [
            pl.col(c).fill_null(df[c].median() or 0.0).alias(c)
            for c in other_numeric_nulls
        ]
        df = df.with_columns(exprs_num)
 
    # ── Cột STRING còn null (edge case) ──────────────────────────────────────
    other_str_nulls = [
        c for c in df.select(cs.string()).columns
        if df[c].null_count() > 0 and c not in DATE_COLS_RAW
    ]
    if other_str_nulls:
        print(f"\n  Các cột chuỗi còn null: {other_str_nulls}")
        df = df.with_columns([
            pl.col(c).fill_null("Unknown") for c in other_str_nulls
        ])
 
    # ── Kiểm tra lại ──────────────────────────────────────────────────────────
    remaining = df.select(
        [c for c in df.columns if c not in DATE_COLS_RAW]
    ).select(pl.all().null_count()).sum_horizontal().item()
    print(f"\n✓ Null còn lại trong các cột NON-DATE: {remaining:,}")
    return df
 
 
print("\nĐang xử lý null còn lại...")
acc_df = fix_remaining_nulls(acc_df)


Đang xử lý null còn lại...

  Giữ nguyên null ở 4 cột date gốc: ['earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d', 'issue_d']
  (Null này vô hại vì các feature phái sinh từ chúng sẽ được fill riêng)

✓ Null còn lại trong các cột NON-DATE: 0


# 3. Vấn đề Data Leakage

In [22]:
# ═══════════════════════════════════════════════════════════════════════════════
# [2] PHÁT HIỆN VÀ LOẠI BỎ DATA LEAKAGE COLUMNS
# ═══════════════════════════════════════════════════════════════════════════════
"""
Data Leakage trong bài toán tín dụng:
──────────────────────────────────────────────────────────────────────────────
Mục tiêu: Dự đoán xem một khoản vay có bị DEFAULT hay không tại thời điểm
phê duyệt (issue_d). Mô hình chỉ được biết các thông tin CÓ SẴN tại thời
điểm đó.
 
Phân loại leakage:
 
  [A] TARGET LEAKAGE (Post-outcome variables):
      Thông tin chỉ có ĐƯỢC SAU KHI kết quả vay đã xảy ra.
      Ví dụ: total_pymnt (tổng tiền đã trả) → chỉ biết sau khi vay kết thúc.
 
  [B] TEMPORAL LEAKAGE (Forward-looking aggregates):
      Thông tin được tính dựa trên dữ liệu từ TƯƠNG LAI.
      Đã được xử lý ở [1] (time_aware_impute).
 
  [C] PROXY LEAKAGE (High-correlation với target):
      Cột mà giá trị của nó "tiết lộ" gần như hoàn toàn kết quả.
      Ví dụ: loan_status (chính là target) không được dùng làm feature.
 
Tài liệu tham khảo:
  • Kaufman et al. (2012). "Leakage in Data Mining: Formulation, Detection,
    and Avoidance." ACM TKDD 6(4). — Bài báo nền tảng về data leakage.
  • Kapoor & Narayanan (2023). "Leakage and the Reproducibility Crisis in
    ML-based Science." Patterns 4(9). — Phân tích leakage trong nghiên cứu.
──────────────────────────────────────────────────────────────────────────────
"""

'\nData Leakage trong bài toán tín dụng:\n──────────────────────────────────────────────────────────────────────────────\nMục tiêu: Dự đoán xem một khoản vay có bị DEFAULT hay không tại thời điểm\nphê duyệt (issue_d). Mô hình chỉ được biết các thông tin CÓ SẴN tại thời\nđiểm đó.\n \nPhân loại leakage:\n \n  [A] TARGET LEAKAGE (Post-outcome variables):\n      Thông tin chỉ có ĐƯỢC SAU KHI kết quả vay đã xảy ra.\n      Ví dụ: total_pymnt (tổng tiền đã trả) → chỉ biết sau khi vay kết thúc.\n \n  [B] TEMPORAL LEAKAGE (Forward-looking aggregates):\n      Thông tin được tính dựa trên dữ liệu từ TƯƠNG LAI.\n      Đã được xử lý ở [1] (time_aware_impute).\n \n  [C] PROXY LEAKAGE (High-correlation với target):\n      Cột mà giá trị của nó "tiết lộ" gần như hoàn toàn kết quả.\n      Ví dụ: loan_status (chính là target) không được dùng làm feature.\n \nTài liệu tham khảo:\n  • Kaufman et al. (2012). "Leakage in Data Mining: Formulation, Detection,\n    and Avoidance." ACM TKDD 6(4). — Bài báo nền 

In [23]:
# ── [A] TARGET LEAKAGE: Biến chỉ có sau khi kết quả vay đã xảy ra ────────────
POST_ORIGINATION_COLS = [
    # Thông tin thanh toán (chỉ biết sau khi vay bắt đầu thanh toán)
    "total_pymnt",           # Tổng tiền đã thu được
    "total_pymnt_inv",       # Tổng tiền nhà đầu tư thu được
    "total_rec_prncp",       # Tổng gốc đã thu
    "total_rec_int",         # Tổng lãi đã thu
    "total_rec_late_fee",    # Phí phạt trả muộn đã thu
    "recoveries",            # Tiền thu hồi sau khi charge-off
    "collection_recovery_fee",  # Phí thu hồi nợ
    "last_pymnt_amnt",       # Số tiền trả lần cuối
    "out_prncp",             # Dư nợ gốc hiện tại (thay đổi theo thời gian)
    "out_prncp_inv",         # Dư nợ gốc nhà đầu tư hiện tại
 
    # Thông tin ngày tháng post-origination
    "last_pymnt_d",          # Ngày trả tiền gần nhất
    "last_credit_pull_d",    # Ngày kiểm tra tín dụng gần nhất
    "next_pymnt_d",          # Ngày trả tiền kế tiếp (đã bị drop ở bước > 70% null)
 
    # Feature phái sinh từ ngày tháng post-origination
    "days_since_last_pymnt_d",       # Phái sinh từ last_pymnt_d
    "days_since_last_credit_pull_d", # Phái sinh từ last_credit_pull_d
 
    # FICO score tại thời điểm HIỆN TẠI (thay đổi sau origination)
    # Lưu ý: fico_range_low / high tại origination thì GIỮ LẠI (hợp lệ)
    "last_fico_range_high",  # FICO hiện tại, không phải tại thời điểm phê duyệt
    "last_fico_range_low",   # Tương tự
]
 
# ── [B] TARGET ITSELF: Cột mục tiêu ──────────────────────────────────────────
# loan_status là TARGET, không phải feature → xử lý riêng (encode, không drop)
TARGET_COL = "loan_status"
 
# ── [C] PROXY / ID COLUMNS: Không có giá trị dự đoán hoặc gây leakage ────────
PROXY_OR_ID_COLS = [
    "id",               # Định danh duy nhất, không có giá trị dự đoán
    "url",              # URL của khoản vay trên Lending Club
    "emp_title",        # Chức danh nghề nghiệp - quá nhiều unique values, không chuẩn
    "title",            # Tiêu đề khoản vay tự điền, quá nhiều unique values
    # Ghi chú: zip_code_prefix đã được xử lý, có thể giữ hoặc drop tùy phân tích
]
 
# ── [D] ADMINISTRATIVE / REDUNDANT COLUMNS: ──────────────────────────────────
ADMIN_COLS = [
    "policy_code",      # Chỉ có 1 giá trị duy nhất (= 1.0 cho tất cả)
    "pymnt_plan",       # Chỉ có 'n' cho tất cả (không phân biệt)
    "funded_amnt",      # = loan_amnt trong hầu hết trường hợp (redundant)
    "funded_amnt_inv",  # Tương tự
    "hardship_flag",    # 'N' cho hầu hết, leakage nếu 'Y' (post-origination)
    "debt_settlement_flag",  # Tương tự
    "disbursement_method",   # 'Cash' cho hầu hết (gần như hằng số)
]

In [24]:
def remove_leakage_columns(
    df: pl.DataFrame,
    verbose: bool = True,
) -> pl.DataFrame:
    """
    Loại bỏ các cột gây Data Leakage, giữ lại cột mục tiêu.
 
    Trả về (df_clean, target_series) để dùng cho bước mô hình hóa.
    """
    all_cols_to_drop = []
 
    categories = {
        "A. Post-origination (target leakage)": POST_ORIGINATION_COLS,
        "C. Proxy / ID": PROXY_OR_ID_COLS,
        "D. Administrative / Redundant": ADMIN_COLS,
    }
 
    for category, cols in categories.items():
        existing = [c for c in cols if c in df.columns]
        if verbose and existing:
            print(f"\n  [{category}] — Xóa {len(existing)} cột:")
            for c in existing:
                unique_vals = df[c].n_unique()
                print(f"    - {c:<45} (unique: {unique_vals:,})")
        all_cols_to_drop.extend(existing)
 
    # Kiểm tra xem TARGET có tồn tại không
    if TARGET_COL not in df.columns:
        print(f"\n  CẢNH BÁO: Cột mục tiêu '{TARGET_COL}' không tồn tại trong df!")
    else:
        print(f"\n  Giữ lại cột mục tiêu: '{TARGET_COL}'")
 
    # Loại bỏ trùng lặp trong danh sách drop
    all_cols_to_drop = list(set(all_cols_to_drop))
    df = df.drop([c for c in all_cols_to_drop if c in df.columns])
 
    print(f"\n✓ Đã loại bỏ {len(all_cols_to_drop)} cột leakage.")
    print(f"  Shape còn lại: {df.shape}")
    return df

In [25]:
def encode_target(df: pl.DataFrame, target_col: str = "loan_status") -> pl.DataFrame:
    """
    Encode cột mục tiêu loan_status thành nhị phân:
        0 = Fully Paid (vay tốt)
        1 = Charged Off / Default (vay xấu)
    Loại bỏ các trạng thái trung gian (Current, Late, ...) để
    bài toán classification được rõ ràng.
 
    Tham khảo: Cách xử lý phổ biến trong các nghiên cứu về
    Lending Club (He et al. 2018; Byanjankar et al. 2015).
    """
    GOOD_STATUS = ["Fully Paid"]
    BAD_STATUS  = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off"]
 
    df = df.filter(pl.col(target_col).is_in(GOOD_STATUS + BAD_STATUS))
 
    df = df.with_columns(
        pl.when(pl.col(target_col).is_in(BAD_STATUS))
        .then(1)
        .otherwise(0)
        .cast(pl.Int8)
        .alias("target")
    ).drop(target_col)
 
    default_rate = df["target"].mean() * 100
    print(f"\n  Phân phối target sau encode:")
    print(f"    Default (1): {df['target'].sum():,}  ({default_rate:.1f}%)")
    print(f"    Good    (0): {(df.height - df['target'].sum()):,}  ({100-default_rate:.1f}%)")
 
    return df

In [26]:
print("\n" + "=" * 60)
print("[2] PHÁT HIỆN VÀ XỬ LÝ DATA LEAKAGE COLUMNS")
print("=" * 60)
 
acc_df = remove_leakage_columns(acc_df, verbose=True)
 
print("\nĐang encode cột mục tiêu...")
acc_df = encode_target(acc_df, target_col="loan_status")


[2] PHÁT HIỆN VÀ XỬ LÝ DATA LEAKAGE COLUMNS

  [A. Post-origination (target leakage)] — Xóa 17 cột:
    - total_pymnt                                   (unique: 1,553,662)
    - total_pymnt_inv                               (unique: 1,235,716)
    - total_rec_prncp                               (unique: 484,200)
    - total_rec_int                                 (unique: 635,921)
    - total_rec_late_fee                            (unique: 18,375)
    - recoveries                                    (unique: 132,777)
    - collection_recovery_fee                       (unique: 146,222)
    - last_pymnt_amnt                               (unique: 704,467)
    - out_prncp                                     (unique: 356,141)
    - out_prncp_inv                                 (unique: 368,481)
    - last_pymnt_d                                  (unique: 137)
    - last_credit_pull_d                            (unique: 142)
    - next_pymnt_d                                  (unique: 107

**Vấn đề**: Imbalance data

# Kiểm tra thêm: Xem còn cột nào tương quan cao với Target không? 

In [27]:
# ── Kiểm tra tự động: Cột nào còn có tương quan nghi ngờ với target? ─────────
def check_suspicious_correlation(df: pl.DataFrame, target: str = "target", top_n: int = 10):
    """
    Kiểm tra tương quan Pearson giữa các cột số và target.
    Cột có |correlation| > 0.5 cần xem xét kỹ: có thể là proxy leakage.
    Đây là bước KIỂM TRA, không tự động drop.
    """
    numeric_cols = [c for c in df.select(cs.numeric()).columns if c != target]
    corr_list = []
    for c in numeric_cols:
        try:
            corr = df.select(
                pl.corr(pl.col(c).cast(pl.Float64), pl.col(target).cast(pl.Float64))
            ).item()
            if corr is not None:
                corr_list.append({"Column": c, "Correlation": round(abs(corr), 4)})
        except Exception:
            continue
 
    corr_df = pl.DataFrame(corr_list).sort("Correlation", descending=True)
    print(f"\nTop {top_n} cột có tương quan cao nhất với target:")
    print(corr_df.head(top_n))
    suspicious = corr_df.filter(pl.col("Correlation") > 0.5)
    if suspicious.height > 0:
        print(f"\nCẢNH BÁO: {suspicious.height} cột có |corr| > 0.5 với target:")
        print(suspicious)
        print("→ Cần kiểm tra thủ công xem chúng có phải post-origination data không.")
    return corr_df
 
 
print("\nKiểm tra tương quan để phát hiện proxy leakage...")
corr_report = check_suspicious_correlation(acc_df, target="target", top_n=15)


Kiểm tra tương quan để phát hiện proxy leakage...

Top 15 cột có tương quan cao nhất với target:
shape: (15, 2)
┌──────────────────────┬─────────────┐
│ Column               ┆ Correlation │
│ ---                  ┆ ---         │
│ str                  ┆ f64         │
╞══════════════════════╪═════════════╡
│ int_rate             ┆ 0.2592      │
│ term_months          ┆ 0.1756      │
│ fico_range_low       ┆ 0.1324      │
│ fico_range_high      ┆ 0.1324      │
│ dti                  ┆ 0.1077      │
│ acc_open_past_24mths ┆ 0.1016      │
│ num_tl_op_past_12m   ┆ 0.0852      │
│ bc_open_to_buy       ┆ 0.0785      │
│ tot_hi_cred_lim      ┆ 0.078       │
│ avg_cur_bal          ┆ 0.0748      │
│ mort_acc             ┆ 0.0729      │
│ num_actv_rev_tl      ┆ 0.0708      │
│ total_bc_limit       ┆ 0.0697      │
│ num_rev_tl_bal_gt_0  ┆ 0.069       │
│ tot_cur_bal          ┆ 0.0689      │
└──────────────────────┴─────────────┘


**Không ảnh hưởng vì tương quan thấp**

In [28]:
# Kiểm tra cuối
print("=" * 60)
print("SHAPE CUỐI CÙNG (sau tất cả xử lý):")
print(f"  Rows: {acc_df.height:,}")
print(f"  Cols: {acc_df.width}")
total_null = acc_df.select(
    [c for c in acc_df.columns
     if c not in ["earliest_cr_line", "last_pymnt_d", "last_credit_pull_d", "issue_d"]]
).select(pl.all().null_count()).sum_horizontal().item()
print(f"  Null còn lại (cột non-date): {total_null:,}")
print("=" * 60)

SHAPE CUỐI CÙNG (sau tất cả xử lý):
  Rows: 1,346,111
  Cols: 85
  Null còn lại (cột non-date): 0
